# Concept Intervention Playground

This notebook lets you inspect one CUB test image at a time, compare:
- GT class
- GroundingDINO GT concepts
- predicted concept activations
- predicted class logits

and then manually edit concept activations to see whether the final class output changes.

This notebook is configured for the **pod workspace paths** by default, because the checkpoints live under `/workspace/SAVLGCBM/...`.


## What this notebook does

1. Load a trained CBM checkpoint (`LF`, `VLG`, `SALF`, or `SAVLG`).
2. Load one CUB **test** image with the same GroundingDINO-derived GT concepts used elsewhere in the repo.
3. Show:
   - raw image
   - GT annotation overlay
   - GT concept list
   - top-k predicted concepts
   - top-k predicted classes
4. Let you intervene in two ways:
   - **binary overrides**: set a concept to `0/1` and map it into the model's intervention space
   - **space overrides**: directly write a value into the concept neuron
5. Recompute the final class prediction after intervention.

The binary override path matches the classic CBM-style intervention more closely.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data import utils as data_utils
from data.concept_dataset import _unwrap_dataset_indices
from methods.common import load_run_info
from scripts.evaluate_concept_interventions import (
    _build_test_loader,
    _get_batch_model_state,
    _lf_state,
    _load_checkpoint_args,
    _load_concepts,
    _salf_state,
    _savlg_state,
    _vlg_state,
)


In [ ]:
# Default pod-side paths. Change these if you want a different checkpoint.
LOAD_PATH = "/workspace/SAVLGCBM/saved_models/cub/savlg_cbm_cub_2026_04_07_23_45_15-1"
ANNOTATION_DIR = "/workspace/SAVLGCBM/annotations"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Eval slice. The notebook works on the CUB test set.
NUM_IMAGES = 500
BATCH_SIZE = 128
NUM_WORKERS = 8

# SAVLG only. Ignored for LF/VLG/SALF.
SAVLG_SCORE_SOURCE = "final"

# Interactive inspection defaults.
CASE_INDEX = 16              # index inside the selected test slice [0, NUM_IMAGES)
TOPK_CONCEPTS = 20
TOPK_CLASSES = 5

# Optional alternatives you can paste into LOAD_PATH:
# VLG resnet18: /workspace/SAVLGCBM/saved_models/cub/cub_cbm_2026_04_06_06_16_10
# SAVLG resnet50 conv45: /workspace/SAVLGCBM/saved_models/savlg_cbm_cub_2026_04_09_20_39_53


In [ ]:
run_info = load_run_info(LOAD_PATH)
args = _load_checkpoint_args(LOAD_PATH, DEVICE, ANNOTATION_DIR)
model_name = run_info.get("model_name", "vlg_cbm")
concepts = _load_concepts(LOAD_PATH)
class_names = data_utils.get_classes(args.dataset)
concept_to_idx = {concept: idx for idx, concept in enumerate(concepts)}

test_loader = _build_test_loader(
    model_name,
    LOAD_PATH,
    args,
    concepts,
    BATCH_SIZE,
    NUM_WORKERS,
    NUM_IMAGES,
)
base_dataset, subset_indices = _unwrap_dataset_indices(test_loader.dataset)

print({
    "model_name": model_name,
    "dataset": args.dataset,
    "device": args.device,
    "num_images": len(test_loader.dataset),
    "num_concepts": len(concepts),
})


In [ ]:
def _get_final_layer():
    if model_name == "lf_cbm":
        return _lf_state(LOAD_PATH, args).final
    if model_name == "vlg_cbm":
        return _vlg_state(LOAD_PATH, args)[3]
    if model_name == "salf_cbm":
        return _salf_state(LOAD_PATH, args)[4]
    if model_name == "savlg_cbm":
        return _savlg_state(LOAD_PATH, args)[4]
    raise NotImplementedError(model_name)


def _get_annotations(dataset_index: int):
    if hasattr(base_dataset, "_normalize_annotations") and hasattr(base_dataset, "_load_raw_data"):
        return base_dataset._normalize_annotations(base_dataset._load_raw_data(dataset_index))
    return base_dataset.get_annotations(dataset_index)


def _class_table(logits: torch.Tensor, topk: int = TOPK_CLASSES) -> pd.DataFrame:
    probs = logits.softmax(dim=0)
    values, indices = logits.topk(k=min(topk, logits.numel()))
    rows = []
    for rank, (idx, logit) in enumerate(zip(indices.tolist(), values.tolist()), start=1):
        rows.append({
            "rank": rank,
            "class": class_names[idx],
            "logit": float(logit),
            "prob": float(probs[idx].item()),
        })
    return pd.DataFrame(rows)


def _concept_table(concept_space: torch.Tensor, gt_binary: torch.Tensor, topk: int = TOPK_CONCEPTS) -> pd.DataFrame:
    values, indices = concept_space.topk(k=min(topk, concept_space.numel()))
    rows = []
    for rank, (idx, score) in enumerate(zip(indices.tolist(), values.tolist()), start=1):
        rows.append({
            "rank": rank,
            "concept": concepts[idx],
            "score": float(score),
            "is_gt": bool(gt_binary[idx].item() > 0),
        })
    return pd.DataFrame(rows)


def _gt_rank_table(concept_space: torch.Tensor, gt_binary: torch.Tensor) -> pd.DataFrame:
    order = concept_space.argsort(descending=True)
    ranks = torch.empty_like(order)
    positions = torch.arange(1, concept_space.numel() + 1, device=concept_space.device)
    ranks.scatter_(0, order, positions)
    rows = []
    gt_indices = (gt_binary > 0).nonzero(as_tuple=False).flatten().tolist()
    for idx in gt_indices:
        rows.append({
            "concept": concepts[idx],
            "rank": int(ranks[idx].item()),
            "score": float(concept_space[idx].item()),
        })
    rows = sorted(rows, key=lambda x: x["rank"])
    return pd.DataFrame(rows)


def get_case(case_index: int):
    image_tensor, gt_concept_one_hot, target = test_loader.dataset[case_index]
    dataset_index = int(subset_indices[case_index])
    image_pil = base_dataset.get_image_pil(dataset_index).convert("RGB")
    annotations = _get_annotations(dataset_index)
    batch_image = image_tensor.unsqueeze(0)

    concept_space, logits, gt_transform = _get_batch_model_state(model_name, LOAD_PATH, args, batch_image)
    concept_space = concept_space[0].detach()
    logits = logits[0].detach()

    gt_binary = gt_concept_one_hot.float().cpu()
    gt_space = gt_transform(gt_binary.unsqueeze(0).to(args.device))[0].detach()
    final_layer = _get_final_layer()

    return {
        "case_index": int(case_index),
        "dataset_index": dataset_index,
        "image_tensor": image_tensor,
        "image_pil": image_pil,
        "annotations": annotations,
        "target_idx": int(target),
        "target_name": class_names[int(target)],
        "gt_binary": gt_binary,
        "concept_space": concept_space,
        "gt_space": gt_space,
        "gt_transform": gt_transform,
        "logits": logits,
        "pred_idx": int(logits.argmax().item()),
        "pred_name": class_names[int(logits.argmax().item())],
        "final_layer": final_layer,
    }


def show_case(case, topk_concepts: int = TOPK_CONCEPTS, topk_classes: int = TOPK_CLASSES):
    display(Markdown(f"## Case {case['case_index']} (dataset idx {case['dataset_index']})"))
    display(Markdown(f"**GT class:** `{case['target_name']}`  \
**Pred class:** `{case['pred_name']}`"))

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(case["image_pil"])
    ax.axis("off")
    ax.set_title("Raw image")
    plt.show()

    overlay_fig = data_utils.plot_annotations(case["image_pil"], case["annotations"])
    overlay_fig.suptitle("GroundingDINO GT concept boxes", fontsize=12)
    plt.show()

    gt_concepts = [concepts[i] for i in (case["gt_binary"] > 0).nonzero(as_tuple=False).flatten().tolist()]
    display(Markdown("**GT concepts**"))
    display(pd.DataFrame({"concept": gt_concepts}))

    display(Markdown("**GT concept ranks in predicted concept space**"))
    display(_gt_rank_table(case["concept_space"].detach().cpu(), case["gt_binary"].detach().cpu()))

    display(Markdown(f"**Top-{topk_concepts} predicted concepts**"))
    display(_concept_table(case["concept_space"].detach().cpu(), case["gt_binary"].detach().cpu(), topk=topk_concepts))

    display(Markdown(f"**Top-{topk_classes} predicted classes**"))
    display(_class_table(case["logits"].detach().cpu(), topk=topk_classes))


def intervene_case(case, binary_overrides: Dict[str, float] | None = None, space_overrides: Dict[str, float] | None = None):
    binary_overrides = binary_overrides or {}
    space_overrides = space_overrides or {}

    edited_concepts = case["concept_space"].clone()
    edited_binary = case["gt_binary"].clone()

    for concept_name, value in binary_overrides.items():
        idx = concept_to_idx[concept_name]
        edited_binary[idx] = float(value)

    if binary_overrides:
        transformed = case["gt_transform"](edited_binary.unsqueeze(0).to(args.device))[0].detach()
        for concept_name in binary_overrides:
            idx = concept_to_idx[concept_name]
            edited_concepts[idx] = transformed[idx]

    for concept_name, value in space_overrides.items():
        idx = concept_to_idx[concept_name]
        edited_concepts[idx] = float(value)

    with torch.no_grad():
        edited_logits = case["final_layer"](edited_concepts.unsqueeze(0))[0].detach()

    delta = (edited_concepts - case["concept_space"]).detach().cpu()
    changed_idx = delta.abs().argsort(descending=True)
    changed_rows = []
    for idx in changed_idx[:20].tolist():
        if abs(float(delta[idx].item())) < 1e-9:
            continue
        changed_rows.append({
            "concept": concepts[idx],
            "before": float(case["concept_space"][idx].detach().cpu().item()),
            "after": float(edited_concepts[idx].detach().cpu().item()),
            "delta": float(delta[idx].item()),
            "is_gt": bool(case["gt_binary"][idx].item() > 0),
        })

    return {
        "edited_concepts": edited_concepts,
        "edited_logits": edited_logits,
        "pred_idx": int(edited_logits.argmax().item()),
        "pred_name": class_names[int(edited_logits.argmax().item())],
        "changed_concepts": pd.DataFrame(changed_rows),
    }


def show_intervention(case, intervention_result, topk_classes: int = TOPK_CLASSES):
    display(Markdown(f"### After intervention"))
    display(Markdown(f"**Before:** `{case['pred_name']}`  \
**After:** `{intervention_result['pred_name']}`  \
**GT:** `{case['target_name']}`"))

    display(Markdown("**Top predicted classes after intervention**"))
    display(_class_table(intervention_result["edited_logits"].detach().cpu(), topk=topk_classes))

    display(Markdown("**Concept dimensions changed by the intervention**"))
    display(intervention_result["changed_concepts"])


def scan_cases(limit: int = 50, focus_k: int = 10) -> pd.DataFrame:
    rows = []
    for case_index in range(min(limit, len(test_loader.dataset))):
        case = get_case(case_index)
        gt_rank_df = _gt_rank_table(case["concept_space"].detach().cpu(), case["gt_binary"].detach().cpu())
        best_gt_rank = int(gt_rank_df["rank"].min()) if len(gt_rank_df) else None
        rows.append({
            "case_index": case_index,
            "dataset_index": case["dataset_index"],
            "target_name": case["target_name"],
            "pred_name": case["pred_name"],
            "best_gt_rank": best_gt_rank,
            f"hit_at_{focus_k}": best_gt_rank is not None and best_gt_rank <= focus_k,
        })
    return pd.DataFrame(rows).sort_values([f"hit_at_{focus_k}", "best_gt_rank", "case_index"], ascending=[True, False, True])


## Inspect one case

Set `CASE_INDEX` above, then run the next cell. `CASE_INDEX` is the index **inside the selected test slice** (for example, inside the first `500` test images if `NUM_IMAGES=500`).


In [ ]:
case = get_case(CASE_INDEX)
show_case(case, topk_concepts=TOPK_CONCEPTS, topk_classes=TOPK_CLASSES)


## Manual concept intervention

There are two supported override types:

- `binary_overrides`:
  - set concept presence to `0.0` or `1.0`
  - then map that binary value into the model's intervention space
- `space_overrides`:
  - directly set the concept neuron to a numeric value in the model's concept space

Start with `binary_overrides`. That is closer to the classic CBM intervention protocol.


In [ ]:
# Edit these dictionaries and rerun.
# Example:
# binary_overrides = {
#     "yellow beak": 1.0,
#     "white body": 1.0,
# }
# space_overrides = {
#     "long wings": 3.0,
# }

binary_overrides = {}
space_overrides = {}

intervention = intervene_case(
    case,
    binary_overrides=binary_overrides,
    space_overrides=space_overrides,
)
show_intervention(case, intervention, topk_classes=TOPK_CLASSES)


## Optional: search for interesting failure cases

This scans the first few cases in the selected test slice and reports where GT concepts are badly ranked.


In [ ]:
scan_cases(limit=50, focus_k=10).head(20)
